In [1]:
import warnings
warnings.filterwarnings("ignore")

import torch
import monai
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from monai.data import Dataset, DataLoader
from monai.utils import set_determinism
from sklearn.model_selection import train_test_split
from tqdm import tqdm

TensorRT-LLM is not installed. Please install TensorRT-LLM or set TRTLLM_PLUGINS_PATH to the directory containing libnvinfer_plugin_tensorrt_llm.so to use converters for torch.distributed ops


[10/26/2025-12:57:31] [TRT] [W] Functionality provided through tensorrt.plugin module is experimental.


In [2]:
import sys
import os

parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

from src.data.temporal_loader import load_temporal_splits_from_json
from src.data.transforms import build_train_transform, build_eval_transform
from src.data.utils import pad_sequence_collate_fn, TransformSequence

from src.networks.t_unetr import TemporalSpatialEmbedding

device = torch.device("cuda")

In [3]:
img_size = (128, 128, 64)

# Create transform
eval_tf  = build_eval_transform(roi_size=img_size)

In [4]:
# Set seed
seed = 42
set_determinism(seed)

# Assemble per-patient sequences
root = "/scratch/ejf9db/new T1_split"
splits = load_temporal_splits_from_json(root)
X_train, X_val, X_test = splits["train"], splits["val"], splits["test"]

In [5]:
# Create MONAI dataset
transforms = TransformSequence(keys=["images", "labels", "dates"], spatial_transforms=eval_tf)
train_dataset = Dataset(data=X_train, transform=transforms)
val_dataset = Dataset(data=X_val, transform=transforms)
test_dataset = Dataset(data=X_train, transform=transforms)

In [6]:
print(train_dataset[0]["images"].shape)
print(val_dataset[0]["images"].shape)

torch.Size([2, 1, 128, 128, 64])
torch.Size([2, 1, 128, 128, 64])


# Extract Embeddings

Run SpatialTemporalEmbeddings

In [7]:
in_channels: int = 1
out_channels: int = 1
img_size = img_size
patch_size: int = 16
feature_size: int = 16
embed_dim: int = 768
mlp_dim: int = 3072
num_heads: int = 12
temporal_depth: int = 4
use_temporal_mlp: bool = True
use_temporal_encoder = "position"
proj_type: str = "conv"
norm_name: tuple | str = "instance"
spatial_dims: int = 3

net = TemporalSpatialEmbedding(
    in_channels=in_channels,
    img_size=img_size,
    patch_size=patch_size,
    num_layers=12,
    embed_dim=embed_dim,
    mlp_dim=mlp_dim,
    num_heads=num_heads,
    temporal_depth=4,
    use_temporal_mlp=use_temporal_mlp,
    use_temporal_encoder=use_temporal_encoder,
    proj_type=proj_type,
    dropout=0.1,
    spatial_dims=spatial_dims
)

net.load_state_dict(torch.load("epochs200_td4_position_lr1e4_tverskyce_warmup_lam10_128x128x64_1020_pretrain_embedding.pth"))

<All keys matched successfully>

In [9]:
# Train data
path = os.getcwd()
new_path = os.path.join(path, "train_data_embeddings")
os.makedirs(new_path, exist_ok=True)

net.eval()
net.to(device)

with torch.no_grad():
    for idx, (batch, raw) in enumerate(zip(train_dataset, X_train)):
        pat_id, scan_ids = raw["patient_id"], raw["scan_ids"]
        images, dates = (
            batch["images"].to(device),  
            batch["dates"].to(device)
        )
        
        seq_lengths = torch.tensor([len(images)], dtype=torch.long).to(device)
        
        images, seq_lengths, dates = images.unsqueeze(0), seq_lengths.unsqueeze(0), dates.unsqueeze(0)
        
        T = images.shape[1]
        
        print(f"embedding data {idx+1}/{len(train_dataset)}")
        
        for i in range(T-1):
            x, ds = images[:, :i+1], dates[:, :i+2]
            
            adjusted_seq_lengths = torch.minimum(seq_lengths, torch.full_like(seq_lengths, i+1)).clamp_min(1.0)
            
            output, _ = net(x, adjusted_seq_lengths, ds)
            
            # Get the mean
            aggregated = torch.mean(output, dim=1)
            
            #print(f"successfully embedded {i}/{T-1} inputs")
            #print("saving aggregated outputs")
            
            # Save
            aggregated = aggregated.squeeze(0).cpu().numpy()
            df = pd.DataFrame(aggregated)
            fn = f"embedding_{pat_id}_{scan_ids[i+1]}.csv"
            file_path = os.path.join(new_path, fn)
            df.to_csv(file_path)
        
        print(f"successfully embedded data {idx+1}")

embedding data 1/109
successfully embedded data 1
embedding data 2/109
successfully embedded data 2
embedding data 3/109
successfully embedded data 3
embedding data 4/109
successfully embedded data 4
embedding data 5/109
successfully embedded data 5
embedding data 6/109
successfully embedded data 6
embedding data 7/109
successfully embedded data 7
embedding data 8/109
successfully embedded data 8
embedding data 9/109
successfully embedded data 9
embedding data 10/109
successfully embedded data 10
embedding data 11/109
successfully embedded data 11
embedding data 12/109
successfully embedded data 12
embedding data 13/109
successfully embedded data 13
embedding data 14/109
successfully embedded data 14
embedding data 15/109
successfully embedded data 15
embedding data 16/109
successfully embedded data 16
embedding data 17/109
successfully embedded data 17
embedding data 18/109
successfully embedded data 18
embedding data 19/109
successfully embedded data 19
embedding data 20/109
successf

In [10]:
# Val data
path = os.getcwd()
new_path = os.path.join(path, "val_data_embeddings")
os.makedirs(new_path, exist_ok=True)

net.eval()
net.to(device)

with torch.no_grad():
    for idx, (batch, raw) in enumerate(zip(val_dataset, X_val)):
        pat_id, scan_ids = raw["patient_id"], raw["scan_ids"]
        images, dates = (
            batch["images"].to(device),  
            batch["dates"].to(device)
        )
        
        seq_lengths = torch.tensor([len(images)], dtype=torch.long).to(device)
        
        images, seq_lengths, dates = images.unsqueeze(0), seq_lengths.unsqueeze(0), dates.unsqueeze(0)
        
        T = images.shape[1]
        
        print(f"embedding data {idx+1}/{len(val_dataset)}")
        
        for i in range(T-1):
            x, ds = images[:, :i+1], dates[:, :i+2]
            
            adjusted_seq_lengths = torch.minimum(seq_lengths, torch.full_like(seq_lengths, i+1)).clamp_min(1.0)
            
            output, _ = net(x, adjusted_seq_lengths, ds)
            
            # Get the mean
            aggregated = torch.mean(output, dim=1)
            
            #print(f"successfully embedded {i}/{T-1} inputs")
            #print("saving aggregated outputs")
            
            # Save
            aggregated = aggregated.squeeze(0).cpu().numpy()
            df = pd.DataFrame(aggregated)
            fn = f"embedding_{pat_id}_{scan_ids[i+1]}.csv"
            file_path = os.path.join(new_path, fn)
            df.to_csv(file_path)
        
        print(f"successfully embedded data {idx+1}")

embedding data 1/23
successfully embedded data 1
embedding data 2/23
successfully embedded data 2
embedding data 3/23
successfully embedded data 3
embedding data 4/23
successfully embedded data 4
embedding data 5/23
successfully embedded data 5
embedding data 6/23
successfully embedded data 6
embedding data 7/23
successfully embedded data 7
embedding data 8/23
successfully embedded data 8
embedding data 9/23
successfully embedded data 9
embedding data 10/23
successfully embedded data 10
embedding data 11/23
successfully embedded data 11
embedding data 12/23
successfully embedded data 12
embedding data 13/23
successfully embedded data 13
embedding data 14/23
successfully embedded data 14
embedding data 15/23
successfully embedded data 15
embedding data 16/23
successfully embedded data 16
embedding data 17/23
successfully embedded data 17
embedding data 18/23
successfully embedded data 18
embedding data 19/23
successfully embedded data 19
embedding data 20/23
successfully embedded data 2